In [1]:
%matplotlib inline
from beamforming import *
from plot_utils import *
from util import *
from sensor_array import *
from psd_estimation import *
from feature_extraction import *

import os
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, FloatSlider, Output, Label, Dropdown, FloatLogSlider, Text, Button, Layout
from IPython.display import display, clear_output, Audio
import json

In [2]:
def make_envelope(x, sr, max_points=250):
    """Downsample x into min/max envelope for fast plotting."""
    n = len(x)
    if n == 0:
        return np.array([0.0]), np.array([0.0]), np.array([0.0])
    if n <= max_points:
        t = np.arange(n) / sr
        return t, x, x
    hop = int(np.ceil(n / max_points))
    idx = np.arange(0, n, hop)
    # Use ufunc.reduceat for fast min/max per chunk
    y_min = np.minimum.reduceat(x, idx)
    y_max = np.maximum.reduceat(x, idx)
    t = idx / sr
    return t, -np.abs(y_min)**0.5, np.abs(y_max)**0.5

In [4]:
fig_width = 11
c = 344
duration = 5
N = WINDOW_LENGTH
step = WINDOW_STEP
window = WINDOW
beamforming_resolution = 100

array = array_circle8()

meas1 = [
    "../samples/meas1_0_10_50_neo.wav",
    "../samples/meas1_60_10_90_neo.wav",
    "../samples/meas1_0_5_50_mini3.wav",
    "../samples/meas1_0_5_80_mavic4.wav",
    "../samples/meas1_ambience.wav"
]

meas2 = [
    "../samples/meas2_0_10_90_neo.wav",
    "../samples/meas2_0_5_50_mini3.wav",
    "../samples/meas2_0_5_80_mavic4.wav",
]

audio_path = meas1[1]
audio, sr = sf.read(audio_path, always_2d=True)
audio = audio.T
total_duration = audio.shape[1] / sr

fmin = FMIN
fmax = FMAX
VN, f, validf = NSTFT, VALID_FREQUENCIES, VALID_FREQUENCY_INDICES

look_directions, aziv, zeniv = create_look_directions(beamforming_resolution, 90 / todeg)
distance_vectors = create_distance_vectors(look_directions, array.positions)
plot_points = (aziv.reshape((beamforming_resolution, beamforming_resolution)),
               zeniv.reshape((beamforming_resolution, beamforming_resolution)))

t_env, y_min_env, y_max_env = make_envelope(audio[0], sr)
y_abs_max = max(1e-9, float(np.max(np.abs([y_min_env, y_max_env]))))

w_azi = FloatSlider(
    value=180, min=0, max=360,
    step=0.1,
    description="Azi", continuous_update=True, readout_format=".1f"
)
w_zeni = FloatSlider(
    value=0, min=0, max=90,
    step=0.1,
    description="Zeni", continuous_update=True, readout_format=".1f"
)
w_freq = FloatSlider(
    value=1000, min=fmin, max=fmax,
    step=1,
    description="Freq (Hz)", continuous_update=True, readout_format=".2f"
)
w_start = FloatSlider(
    value=0.0, min=0.0, max=max(0.0, total_duration - 0.01),
    step=max(0.001, total_duration / 1000.0),
    description="Start (s)", continuous_update=True, readout_format=".2f"
)
w_feat = Dropdown(options=["STFT", "MFCC", "MVDR", "NOBEC"], value="MFCC", description="Feat:")

start_button = Button(
    description="Start",
    button_style="info",
)
stop_button = Button(
    description="Stop",
    button_style="info",
)
path_box = Text(
    value="../dataset/"+os.path.splitext(os.path.basename(audio_path))[0],
    description="File:",
    layout=Layout(width="400px")
)
autosave_button = Button(
    description="Sample",
    button_style="warning",
    icon="save"
)
save_button = Button(
    description="Save",
    button_style="success",
    icon="save"
)

w_azi.layout.width = f"{fig_width * 80 * 0.5}px"
w_zeni.layout.width = f"{fig_width * 80 * 0.5}px"

w_freq.layout.width = f"{fig_width * 80 * 1.15}px"
w_start.layout.width = f"{fig_width * 80 * 1.15}px"

out1 = Output()
out2 = Output()

class Sampler:
    start = 0
    end = duration
    s_idx = 0
    e_idx = sr * duration

sampler = Sampler()

def get_time_indices():
    start = max(0.0, min(w_start.value, total_duration))
    end = start + duration
    end = min(end, total_duration)
    start = end - duration

    s_idx = int(round(start * sr))
    e_idx = int(round(end * sr))
    s_idx = max(0, min(s_idx, audio.shape[1]-1))
    e_idx = max(s_idx + 1, min(e_idx, audio.shape[1]))

    return start, end, s_idx, e_idx

def update_plot(change):
    start, end, s_idx, e_idx = get_time_indices()

    x = audio[:, s_idx:e_idx]
    x += np.random.randn(*x.shape) * 1e-18

    SN, tidx, snaps = stft(x, window, step, validf)

    fspot = np.abs(f - w_freq.value).argmin()

    TW = SN*3//duration
    Sx = np.array([compute_covariance_matrices(snap) for snap in snaps[-TW:]])
    Sx = fb_average(Sx, array.positions)
    Sx = time_average(Sx, smoothing_alpha(sr*0.1, step, 0.5))

    k = 2 * np.pi * f / c
    kf = np.array([[k[fspot]]])
    steering_vectors = create_steering_vectors(distance_vectors, kf)

    target_udir = polar_to_unitdir(w_azi.value / todeg, w_zeni.value / todeg)
    target_steer = create_steering_vectors(create_distance_vectors(target_udir[None,:], array.positions), k[:,None])

    Sxf = np.repeat(Sx[-1][fspot][None,:,:], beamforming_resolution**2, axis=0)
    beamforming_params = PSDEParams(
        array.positions,
        target_udir, kf,
        steering_vectors,
        Sxf)

    params = PSDEParams(
        array.positions,
        target_udir, k,
        target_steer,
        Sx[0])

    beamformer = DAS_PSDE(beamforming_params)
    psd = beamformer.estimate_psd(Sxf)
    psd = psd.reshape(beamforming_resolution, beamforming_resolution)

    das_estimator = DAS_PSDE(params)
    nobec_estimator = NOBEC_PSDE(db2lin(10), db2lin(-20))
    das_psd = np.array([das_estimator.estimate_psd(s) for s in Sx[-FEATURE_LENGTH:]])
    nobec_psd = np.array([nobec_estimator.update_and_estimate(params, s) for s in Sx[-FEATURE_LENGTH:]])
    # nobec_psd = time_average(nobec_psd, smoothing_alpha(sr*0.1, step, 0.05))

    nmfcc = 24
    if w_feat.value == "MFCC":
        das_feature = extract_mfcc_feature(das_psd, sr, fmin, fmax, N, validf, nmfcc)
        nobec_feature = extract_mfcc_feature(nobec_psd, sr, fmin, fmax, N, validf, nmfcc)
    else:
        das_feature = extract_stft_feature(das_psd)
        nobec_feature = extract_stft_feature(nobec_psd)

    with out2:
        clear_output(wait=True)
        _, (ax_top) = plt.subplots(
            1, 1, figsize=(fig_width, fig_width * 0.1),
        )

        # Minimap
        ax_top.fill_between(t_env, y_min_env, y_max_env, color="0.6", alpha=0.6, linewidth=0)
        ax_top.plot([0, total_duration], [0, 0], color="0.2", linewidth=0.5)  # zero line
        ax_top.axvline(sampler.start, color="black", linestyle="-", linewidth=1.5)
        ax_top.axvline(sampler.end, color="gray", linestyle="-", linewidth=1.5)
        ax_top.axvline(start, color="C1", linestyle="--", linewidth=1.5)
        ax_top.axvline(end, color="C3", linestyle="--", linewidth=1.5)
        ax_top.set_xlim(0, total_duration)
        ax_top.set_ylim(-1.05 * y_abs_max, 1.05 * y_abs_max)
        ax_top.set_axis_off()

    with out2:
        clear_output(wait=True)

        fig = plt.figure(figsize=(fig_width*1.2, fig_width * 0.7))
        ax_spec = fig.add_subplot(221)
        ax_beam = fig.add_subplot(222, projection='polar')
        ax_das = fig.add_subplot(223)
        ax_nobec = fig.add_subplot(224)

        plot_spectrogram(ax_spec, f, tidx/sr, lin2db(snaps[:,0,:]).T)
        ax_spec.axhline(f[fspot]/1000, color="gray", linestyle="--", linewidth=1.5)

        plot_mesh_polar(ax_beam, plot_points, psd)
        ax_beam.scatter(w_azi.value/todeg, w_zeni.value, color='red', marker='x', s=100)

        if w_feat.value == "MFCC":
            plot_mfcc_feature(ax_das, tidx[:FEATURE_LENGTH]/sr, das_feature.T)
            plot_mfcc_feature(ax_nobec, tidx[:FEATURE_LENGTH]/sr, nobec_feature.T)
        elif w_feat.value == "STFT":
            plot_stft_feature(ax_das, f, tidx[:FEATURE_LENGTH]/sr, das_feature.T)
            plot_stft_feature(ax_nobec, f, tidx[:FEATURE_LENGTH]/sr, nobec_feature.T)
        elif w_feat.value == "MVDR":
            mvdr_estimator = MVDR_PSDE(db2lin(10))
            mvdr_psd = np.array([mvdr_estimator.update_and_estimate(params, s) for s in Sx])
            plot_spectrogram(ax_das, f, tidx[:TW]/sr, lin2db(snaps[-TW:,0,:].T), vmax=-20)
            plot_spectrogram(ax_nobec, f, tidx[:TW]/sr, pow2db(mvdr_psd).T, vmax=-20)
        elif w_feat.value == "NOBEC":
            nobec_psd = np.array([nobec_estimator.update_and_estimate(params, s) for s in Sx])
            plot_spectrogram(ax_das, f, tidx[:TW]/sr, lin2db(snaps[-TW:,0,:].T), vmax=-20)
            plot_spectrogram(ax_nobec, f, tidx[:TW]/sr, pow2db(nobec_psd).T, vmax=-20)

        plt.show()

        w_das = wideband_das_weights(target_steer)
        das_snaps = np.array([beamform_wideband(s.T, w_das) for s in snaps])
        sdas = istft(das_snaps, window, tidx, validf)
        ch0 = istft(snaps[:,0,:], window, tidx, validf)
        
        display(Audio(ch0, rate=sr))
        display(Audio(sdas, rate=sr))

# Hook up events and render once
for w in (w_start, w_freq, w_azi, w_zeni, w_feat):
    w.observe(update_plot, names="value")

def set_sampler(tt):
    start, end, s_idx, e_idx = get_time_indices()
    if tt == 0:
        sampler.start = start
        sampler.s_idx = s_idx
    else:
        sampler.end = end
        sampler.e_idx = e_idx
    
    update_plot(None)

def save_sample(_):
    pathbase = path_box.value
    idx = 0
    while os.path.isfile(f"{pathbase}_{idx}.wav"):
        idx += 1
    sf.write(f"{pathbase}_{idx}.wav", audio[:, sampler.s_idx:sampler.e_idx].T, sr)
    target_udir = polar_to_unitdir(w_azi.value / todeg, w_zeni.value / todeg)
    with open(f"{pathbase}_{idx}.json", "w", encoding="utf-8") as fil:
        json.dump(target_udir.tolist(), fil)

def autosave_sample(_):
    pathbase = path_box.value
    _, _, s, _ = get_time_indices()
    sf.write(f"{pathbase}.wav", audio[:, s:s+3*sr].T, sr)
    target_udir = polar_to_unitdir(w_azi.value / todeg, w_zeni.value / todeg)
    with open(f"{pathbase}.json", "w", encoding="utf-8") as fil:
        json.dump(target_udir.tolist(), fil)

start_button.on_click(lambda _: set_sampler(0))
stop_button.on_click(lambda _: set_sampler(1))
save_button.on_click(save_sample)
autosave_button.on_click(autosave_sample)

autosave_button

display(VBox([
    HBox([w_azi, w_zeni, w_feat]),
    HBox([start_button, stop_button, path_box, autosave_button, save_button]),
    w_freq,
    w_start,
    out1,
    out2
]))

update_plot(None)